# Update Sample Points
#### Authors: Mike Bollinger and Madeleine Gallop
#### Project: BOEM - Carolina Long Bay
#### May 31, 2024
#### Feed in CSV from Sample point generation script, Create a feature service and update with sample sites

### Summary: finds two existing feature services (one for stratified and one for regular) in ArcGIS Online (created from Karina's ArcDesktop layers), brings in Survey123 data on a daily basis after surveying (the survey details which sites have been visited) and joins the Survey123 data to the feature services. It then concatonates the two feature services into a single CSV, which will go into the dashboard through the BOEM_Dashboard_Update.ipynb 

# UPDATED v1 06.14.2024
##### This version is being updated by Madeleine Gallop 


<!-- 

The purpose of this script is to keep a record in arcgis that contains a list of all of the points that we plan to sample on this expedition (they can be regular sample points or stratified sample points) and to periodically (at the end of each survey day, for example) update the list of points with data recieved from survey123, so that we know which points have been sampled (Status), when (most recent survey date), and which vehical surveyed it. 

 -->



In [1]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import pandas as pd

# Connect to ArcGIS Online
gis = GIS("home")
#Existing Feature Service ID
ExistingFS = '8cdd28ad7790423089781259c8708034' #If you don't have one of these then run the code, find the FS then fill in the ID



print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

Logged in to NOAA GeoPlatform as karina.urquhart_noaa


In [2]:
####Path to new data####
new_data_dbf = pd.read_csv(r"Karina_Madeleine_Test/NF2406_EM2040_DN164_StratSamp.csv") #r converts a normal string to raw
new_data_dbf.shape

###BOEM
new_columns = ['AUV_Completed_Date', 'ROV_Completed_Date','DropCam_Completed_Date','Status','Most_Recent_Survey_Date']
for col in new_columns:
    new_data_dbf[col] = pd.NaT
new_data_dbf.head()


,SiteID,Lon,Lat,Easting,Northing,Design,DateCreated,SurveyDate,Strata,Depth,...,RoughnessThresh,SlopeThresh,WindowSize,AssignedVehicle,PlotterName,AUV_Completed_Date,ROV_Completed_Date,DropCam_Completed_Date,Status,Most_Recent_Survey_Date
0,1,-77.875487,33.476275,790357.5,3708458.5,Stratified,164.0,164.0,2.0,-28.819548,...,0.032,2.0,7.0,DC,DC_1,NaT,NaT,NaT,NaT,NaT
1,2,-77.877906,33.495186,790069.5,3710549.5,Stratified,164.0,164.0,1.0,-27.744108,...,0.032,2.0,7.0,DC,DC_2,NaT,NaT,NaT,NaT,NaT
2,3,-77.879960,33.520370,789794.5,3713337.5,Stratified,164.0,164.0,2.0,-26.545879,...,0.032,2.0,7.0,AUV,AUV_3,NaT,NaT,NaT,NaT,NaT
3,4,-77.875497,33.472093,790370.5,3707994.5,Stratified,164.0,164.0,1.0,-29.019596,...,0.032,2.0,7.0,AUV,AUV_4,NaT,NaT,NaT,NaT,NaT
4,5,-77.882433,33.516240,789578.5,3712872.5,Stratified,164.0,164.0,1.0,-26.276503,...,0.032,2.0,7.0,AUV,AUV_5,NaT,NaT,NaT,NaT,NaT


In [3]:
####Path to existing data####
# JoinedCSVpath = r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\Joined_Sampling_Points.csv"
# Create a blank csv that later holds joined data
JoinedCSVpath = r"Karina_Madeleine_Test/Joined_Sampling_Points_Test.csv"
# print(JoinedCSVpath)


Run #Create Feature service block if this is the first iteration of data you're adding to AGOL

In [4]:
####Create Feature service
###Path to first file to create the FS
# firstCSVpath = r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\RegSample.csv"
firstCSVpath = r"Karina_Madeleine_Test/RegSample_1km_DN165.csv"
try:
    ssItem = gis.content.get(ExistingFS) 
    print("Feature Service already created, Loaded it")
except Exception:
    print("No Feature Service found, creating one...")
    firstCSV = pd.read_csv(firstCSVpath)
    ###BOEM
    new_columns = ['AUV_Completed_Date', 'ROV_Completed_Date','DropCam_Completed_Date','Status','Most_Recent_Survey_Date']
    for col in new_columns:
        firstCSV[col] = pd.NaT

    #Create Joined CSV with firstCSV + AddedColumns. It just writes a new padded file for the Regular points with the added col into a blank csv
    firstCSV.to_csv(JoinedCSVpath)
    # print(firstCSV.head)
    #set properties
    item_prop = {'title':'BOEM CLB Sample Sites Madeleine Test'} 
    #Add JoinedCSV to FS
    csv_item = gis.content.add(item_properties = item_prop, data = JoinedCSVpath)
    print("test is here")
    csv_item
    #Publish Feature layer
    ssItem = csv_item.publish()
    ssItem
    print("Feature Service created")
print(ssItem)

Feature Service already created, Loaded it
<Item title:"BOEM CLB Sample Sites" type:Feature Layer Collection owner:mike.bollinger_noaa>


In [5]:
#Bring in Older data and add to it
JoinedCSV = pd.read_csv(JoinedCSVpath) #the way I have it the joinedCSV path is for regualr points and the new_data_dbf is the stratified points. we dont really want that though right? we want all the points with some of them filled out, and then just the new points from survey 123
#merge Data together
dataframes = [JoinedCSV,new_data_dbf]
updated_df = pd.concat(dataframes) #is concat the best way to do this? we should prob join by an attributte
updated_df.reset_index(drop=True)
updated_df.shape
#make sure there aren't any duplicate ID's
updated_df.drop_duplicates(subset='SiteID',keep='last', inplace=True)
updated_df.head()

#Update CSV Version of points
updated_df.to_csv(JoinedCSVpath)

In [6]:
#Bring in the original feature layer collection

boem_flayer_collection = FeatureLayerCollection.fromitem(ssItem)
#boem_flayer_collection = FeatureLayerCollection.fromitem(ssITEM)
boem_flayer_collection.properties.layers[0].name

'BOEM_CLB_Sample_Sites'

In [7]:
#the Overwrite
boem_flayer_collection.manager.overwrite(JoinedCSVpath)

Exception: User does not have permissions to access this service
User does not have permissions to access this service
(Error Code: 403)